In [1]:

#df.to_csv("D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/customer_purchases_raw.csv", index=False)

# ============================================================
# Generate 500-row Full Industry Dataset
# Customer Purchase Pattern Analyzer
# Output: ../data/customer_purchase_data.csv
# ============================================================

import os
import numpy as np
import pandas as pd

np.random.seed(42)

# ----------------------------
# Config
# ----------------------------
N_ROWS = 500
ANALYSIS_DATE = pd.Timestamp("2026-06-03")  # fixed date for reproducibility

OUT_PATH = "../data/customer_purchase_data.csv"
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

# ----------------------------
# Master data (realistic lists)
# ----------------------------
first_names = [
    "Aarav","Vivaan","Aditya","Vihaan","Arjun","Sai","Reyansh","Muhammad","Krishna","Ishaan",
    "Emma","Olivia","Ava","Sophia","Isabella","Mia","Amelia","Harper","Evelyn","Abigail",
    "Noah","Liam","Mason","Ethan","Lucas","James","Benjamin","Henry","Alexander","Michael",
    "Charlotte","Ella","Grace","Chloe","Lily","Zoey","Leah","Hannah","Nora","Aria"
]
last_names = [
    "Sharma","Patel","Gupta","Khan","Singh","Mehta","Iyer","Reddy","Das","Jain",
    "Johnson","Williams","Brown","Jones","Miller","Davis","Garcia","Rodriguez","Wilson","Martinez",
    "Anderson","Taylor","Thomas","Moore","Jackson","Martin","Lee","Perez","Thompson","White"
]

cities_regions = [
    ("New York","Northeast"), ("Boston","Northeast"), ("Philadelphia","Northeast"), ("Newark","Northeast"),
    ("Chicago","Midwest"), ("Detroit","Midwest"), ("Minneapolis","Midwest"),
    ("Dallas","South"), ("Austin","South"), ("Miami","South"), ("Orlando","South"), ("Atlanta","South"), ("Nashville","South"),
    ("Los Angeles","West"), ("San Francisco","West"), ("Seattle","West"), ("San Diego","West"), ("Denver","West"), ("Phoenix","West"), ("Portland","West")
]

occupations = [
    "Student","Software Engineer","Data Analyst","Data Scientist","Accountant","Teacher","Nurse","HR Manager",
    "Sales Executive","Marketing Specialist","Product Manager","Operations Manager","Consultant","Business Owner","Designer","Manager","Director"
]

# Product catalog by category
catalog = {
    "Groceries": ["Organic Milk","Whole Wheat Bread","Fresh Eggs","Greek Yogurt","Basmati Rice 5lb","Olive Oil","Coffee Beans","Pasta","Cheese","Fresh Strawberries","Avocados","Chicken Breast","Frozen Pizza","Salmon Fillet"],
    "Beverages": ["Green Tea","Cold Brew Coffee","Sparkling Water","Energy Drink Pack","Orange Juice","Kombucha","Iced Tea","Sports Drink","Herbal Tea","Coffee Latte","Electrolyte Drink","Premium Coffee Pods"],
    "Snacks": ["Protein Bar Pack","Chips Combo","Trail Mix","Cookies Box","Dark Chocolate","Granola","Popcorn","Energy Bar","Nuts Mix","Gummies"],
    "Home Care": ["Laundry Detergent","Trash Bags","Dish Soap","Floor Cleaner","Glass Cleaner","Air Freshener","Disinfectant Spray","Paper Towels","Toilet Cleaner","Sponges Pack"],
    "Personal Care": ["Shampoo","Conditioner","Body Wash","Toothpaste","Deodorant","Hand Sanitizer","Hand Cream","Face Wash","Razor Pack","Vitamin C Tablets"],
    "Beauty": ["Face Serum","Moisturizer","Perfume","Sunscreen","Makeup Kit","Lipstick","Eyeliner","Skincare Set","Body Lotion","Hair Oil"],
    "Baby Products": ["Baby Diapers","Baby Wipes","Baby Formula","Baby Food Jar","Baby Shampoo","Baby Lotion"]
}

categories = list(catalog.keys())

payment_methods = ["Credit Card","Debit Card","UPI/Wallet","Cash","Net Banking"]
purchase_channels = ["Mobile App","Website","Store"]

segments = ["High Value","Young Professionals","Family Shoppers","Budget Buyers"]
loyalty_statuses = ["Gold","Silver","None"]

# ----------------------------
# Helper functions
# ----------------------------
def random_name(size):
    fn = np.random.choice(first_names, size=size)
    ln = np.random.choice(last_names, size=size)
    return [f"{a} {b}" for a,b in zip(fn, ln)]

def assign_segment(age, occupation):
    # Simple rule-based segmentation (realistic enough for portfolio)
    if occupation == "Business Owner" or age >= 45:
        return np.random.choice(["High Value","Family Shoppers"], p=[0.6,0.4])
    if occupation == "Student" or age <= 24:
        return "Budget Buyers"
    # mid career
    return np.random.choice(["Young Professionals","Family Shoppers","High Value"], p=[0.55,0.30,0.15])

def assign_loyalty(segment):
    if segment == "High Value":
        return np.random.choice(["Gold","Silver"], p=[0.75,0.25])
    if segment == "Young Professionals":
        return np.random.choice(["Silver","Gold","None"], p=[0.55,0.20,0.25])
    if segment == "Family Shoppers":
        return np.random.choice(["Silver","None","Gold"], p=[0.55,0.35,0.10])
    return np.random.choice(["None","Silver"], p=[0.80,0.20])

def assign_discount(loyalty, segment):
    # Discount use slightly higher for budget buyers and app users later (we'll also randomize)
    base = 0.35
    if segment == "Budget Buyers":
        base = 0.60
    if loyalty == "Gold":
        base = 0.45
    return "Yes" if np.random.rand() < base else "No"

def assign_purchase_frequency(segment, loyalty):
    # Label only; later you can compute real frequency from transactions too
    if segment == "High Value" and loyalty == "Gold":
        return np.random.choice(["High","Medium"], p=[0.7,0.3])
    if segment == "Budget Buyers":
        return np.random.choice(["High","Medium","Low"], p=[0.45,0.35,0.20])
    if segment == "Family Shoppers":
        return np.random.choice(["Medium","High","Low"], p=[0.55,0.30,0.15])
    return np.random.choice(["Medium","High","Low"], p=[0.50,0.35,0.15])

def get_price_and_qty(category):
    # Category-specific pricing (approx realistic)
    if category == "Groceries":
        unit_price = np.round(np.random.uniform(1.5, 22), 2)
        qty = np.random.choice([1,2,3,4], p=[0.35,0.35,0.20,0.10])
    elif category == "Beverages":
        unit_price = np.round(np.random.uniform(1.0, 20), 2)
        qty = np.random.choice([1,2,3,4,6], p=[0.25,0.30,0.20,0.15,0.10])
    elif category == "Snacks":
        unit_price = np.round(np.random.uniform(1.0, 12), 2)
        qty = np.random.choice([1,2,3,4], p=[0.30,0.35,0.25,0.10])
    elif category == "Home Care":
        unit_price = np.round(np.random.uniform(3.0, 18), 2)
        qty = np.random.choice([1,2,3], p=[0.45,0.40,0.15])
    elif category == "Personal Care":
        unit_price = np.round(np.random.uniform(2.0, 18), 2)
        qty = np.random.choice([1,2,3], p=[0.55,0.35,0.10])
    elif category == "Beauty":
        unit_price = np.round(np.random.uniform(8.0, 45), 2)
        qty = np.random.choice([1,2], p=[0.75,0.25])
    else:  # Baby Products
        unit_price = np.round(np.random.uniform(5.0, 30), 2)
        qty = np.random.choice([1,2,3,4,5], p=[0.35,0.30,0.15,0.10,0.10])
    return int(qty), float(unit_price)

# ----------------------------
# Generate base fields
# ----------------------------
# Customers: create a pool so IDs repeat across transactions (important for frequency/CLV)
N_CUSTOMERS = 180  # enough unique customers for 500 transactions
customer_ids = [f"CUST{str(i).zfill(4)}" for i in range(1, N_CUSTOMERS + 1)]
customer_names = random_name(N_CUSTOMERS)

customer_master = pd.DataFrame({
    "Customer ID": customer_ids,
    "Customer Name": customer_names,
    "Age": np.random.randint(18, 60, size=N_CUSTOMERS),
    "Gender": np.random.choice(["Male","Female"], size=N_CUSTOMERS, p=[0.48,0.52]),
})

# City/Region
city_choices = np.random.choice(len(cities_regions), size=N_CUSTOMERS)
customer_master["City"] = [cities_regions[i][0] for i in city_choices]
customer_master["Region"] = [cities_regions[i][1] for i in city_choices]

# Occupation
customer_master["Occupation"] = np.random.choice(occupations, size=N_CUSTOMERS)

# Segment + Loyalty
customer_master["Customer Segment"] = [
    assign_segment(a, o) for a, o in zip(customer_master["Age"], customer_master["Occupation"])
]
customer_master["Loyalty Status"] = [
    assign_loyalty(s) for s in customer_master["Customer Segment"]
]

# ----------------------------
# Generate transactions (500 rows)
# ----------------------------
# Purchase dates: last ~6 months from analysis date
date_range = pd.date_range(ANALYSIS_DATE - pd.Timedelta(days=180), ANALYSIS_DATE, freq="D")

tx_customer = np.random.choice(customer_master["Customer ID"], size=N_ROWS)

tx = customer_master.set_index("Customer ID").loc[tx_customer].reset_index()

# Products
tx["Product Category"] = np.random.choice(categories, size=N_ROWS)
tx["Product Name"] = [np.random.choice(catalog[c]) for c in tx["Product Category"]]

# Purchase date
tx["Purchase Date"] = np.random.choice(date_range, size=N_ROWS)

# Quantity + Unit Price based on category
qty_price = [get_price_and_qty(c) for c in tx["Product Category"]]
tx["Quantity Purchased"] = [qp[0] for qp in qty_price]
tx["Unit Price"] = [qp[1] for qp in qty_price]

# Payment + Channel
tx["Payment Method"] = np.random.choice(payment_methods, size=N_ROWS, p=[0.40,0.25,0.20,0.10,0.05])
tx["Purchase Channel"] = np.random.choice(purchase_channels, size=N_ROWS, p=[0.45,0.30,0.25])

# Discount + Purchase Frequency labels
tx["Discount Used"] = [
    assign_discount(l, s) for l, s in zip(tx["Loyalty Status"], tx["Customer Segment"])
]
tx["Purchase Frequency"] = [
    assign_purchase_frequency(s, l) for s, l in zip(tx["Customer Segment"], tx["Loyalty Status"])
]

# Total
tx["Total Purchase Value"] = (tx["Quantity Purchased"] * tx["Unit Price"]).round(2)

# Last purchase date: ensure it's >= Purchase Date and <= ANALYSIS_DATE
# We'll simulate as: Purchase Date + random 0..60 days, capped at ANALYSIS_DATE
offset_days = np.random.randint(0, 61, size=N_ROWS)
tx["Last Purchase Date"] = tx["Purchase Date"] + pd.to_timedelta(offset_days, unit="D")
tx.loc[tx["Last Purchase Date"] > ANALYSIS_DATE, "Last Purchase Date"] = ANALYSIS_DATE

# Reorder columns exactly as your project requires
final_cols = [
    "Customer ID","Customer Name","Age","Gender","City","Region","Occupation",
    "Product Category","Product Name","Purchase Date","Quantity Purchased","Unit Price",
    "Total Purchase Value","Payment Method","Purchase Channel","Customer Segment",
    "Loyalty Status","Discount Used","Purchase Frequency","Last Purchase Date"
]
tx = tx[final_cols].copy()

# Make date format consistent (YYYY-MM-DD)
tx["Purchase Date"] = pd.to_datetime(tx["Purchase Date"]).dt.strftime("%Y-%m-%d")
tx["Last Purchase Date"] = pd.to_datetime(tx["Last Purchase Date"]).dt.strftime("%Y-%m-%d")

# Introduce a small amount of realistic data issues (optional, but helps show cleaning skills)
# 1) Missing values
miss_idx = np.random.choice(tx.index, 12, replace=False)
tx.loc[miss_idx[:6], "Unit Price"] = np.nan
tx.loc[miss_idx[6:], "Payment Method"] = np.nan

# 2) A few duplicates
tx = pd.concat([tx, tx.sample(8, random_state=42)], ignore_index=True)

# Save
tx.to_csv(OUT_PATH, index=False)

print("Saved dataset to:", OUT_PATH)
print("Final shape (includes added duplicates):", tx.shape)
tx.head()

(  Customer_ID     Product Purchase_Date  Purchase_Amount Payment_Mode  \
 0        C039     Perfume    2024-10-22           4332.0          UPI   
 1        C029  Sunglasses    2024-10-12           4051.0         Cash   
 2        C015      Wallet    2024-11-09           3640.0  Credit Card   
 3        C043       Jeans    2024-09-15           3966.0   Debit Card   
 4        C008       Shoes    2024-10-24           2526.0          UPI   
 
         City  
 0    Lucknow  
 1  Hyderabad  
 2     Jaipur  
 3    Lucknow  
 4     Jaipur  ,
 (205, 6))